## Sentiment Analysis Methodoloygy
 
The dataset where the models were tested is available here: https://huggingface.co/datasets/ElKulako/stocktwits-crypto <br>
It contains all cryptocurrency-related posts from the StockTwits website (around 1.3M), from 1st of November 2021 to the 15th of June 2022.
 
The provided data was already cleaned and preprocessed, in particular it was removed:
 
- cashtags, hashtags, usernames,
- URLs, crypto wallets,
- Chinese, Korean and Japanese characters,
- (most) UTF-8 encoding issues
- All posts shorter than 4 words
- All duplicate posts
- Spacing and punctuation issues, converted all text to lowercase

Initially, i intended to test on the dataset the following models:
 
- Vader: Lexicon and rule-based sentiment analysis tool that is specifically attuned to sentiments expressed in social media
- Support Vector Machines: A traditional, statistical classifier.
- Random Forest: An ensemble machine learning method utilizing deep decision trees.
- Gemma 2B: Google's lightweight, 2-billion parameter Large Language Model.

All this models were to be benchmarked against CryptoBERT, a specialized Transformer machine learning model trained on 3.2 millions social media posts of various cryptocurrencies. <br>
The benchmark results for CryptoBERT are publicly available, making it an ideal competitor, especially since it was explicitly fine-tuned on a portion of this exact StockTwits dataset.
 
After some testing i decided to drop Gemma 2B and Random forest for two main reasons:
 
- Performance: Preliminary testing showed they did not maintain competitive accuracy compared to the SVM and CryptoBERT.
- Computational Efficiency: At scale the inference time for the LLM and the training complexity of the Random Forest were prohibitively high.

Consequently, the final evaluation focuses on a comparison between VADER, SVM and CryptoBERT.
 
Results & Evaluation
 
VADER and SVM were evaluated on the same exact 83k rows test set of the CryptoBERT paper
 
VADER: Accuracy: 37.34% | Macro F1: 0.3679
 
As expected, the traditional rule-based lexicon struggled. <br>
VADER's reliance on standard English definitions proved inadequate for capturing the nuanced, slang-heavy and emoji-driven nature of cryptocurrency posts.
 
CryptoBERT: Accuracy: 55.60% | Macro F1: 0.5579
 
The specialized transformer established a balanced standard. <br>
Thanks to its deep contextual embeddings pre-trained on crypto-specific text, it successfully managed the complex class imbalances and set a high benchmark for the Macro F1 score.
 
SVM: Accuracy: 59.71% | Macro F1: 0.5916
 
The SVM performed exceptionally well, outperforming CryptoBERT in both categories. <br>
By utilizing a wide range of n-grams (1–5), the model successfully captured the specific "power phrases" and slang that drive sentiment on StockTwits. <br>
It achieved an accuracy 4.11% higher than the state-of-the-art transformer.
 
Conclusion

The results demonstrate that a properly tuned statistical machine learning pipeline can surpass a specialized Transformer model in both raw accuracy and Macro F1 score on this dataset. <br>
The SVM’s superior performance, combined with its significantly lower computational overhead, makes it the most viable and scalable solution for high-frequency sentiment analysis in the cryptocurrency domain.

In [ ]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
import pickle

TARGET_NAMES = ["Bearish", "Neutral", "Bullish"]
LABEL_ORDER  = [0, 1, 2]

vader = SentimentIntensityAnalyzer()

def get_vader_sentiment(text: str) -> int:
    compound = vader.polarity_scores(text)["compound"]
    if compound >= 0.05:
        return 2   # Bullish
    elif compound <= -0.05:
        return 0   # Bearish
    else:
        return 1   # Neutral

def evaluate_model(model_name: str, y_true, y_pred):
    
    print(f" {model_name}")
    print(f"  Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(classification_report(y_true, y_pred, labels=LABEL_ORDER, target_names=TARGET_NAMES, digits=4))

if __name__ == "__main__":

    df_train_1 = pd.read_csv("../data/sentiment/stockwits_train_1.csv")
    df_train_2 = pd.read_csv("../data/sentiment/stockwits_train_2.csv")
    df_test = pd.read_csv("../data/sentiment/stockwits_test.csv")

    df_train = pd.concat([df_train_1, df_train_2], ignore_index=True)
    df_tune = df_train.sample(frac=0.2, random_state=42)
    x_tune, y_tune = df_tune["text"].tolist(), df_tune["label"].tolist()

    
    x_train, y_train = df_train["text"].tolist(), df_train["label"].tolist()
    x_test, y_test = df_test["text"].tolist(), df_test["label"].tolist()
    
    results_df = pd.DataFrame({"text": x_test, "label": y_test})

    print("Running VADER...")
    results_df["vader_pred"] = results_df["text"].apply(get_vader_sentiment)
    
    print("Training & running SVM...")
    svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LinearSVC(class_weight="balanced", dual=False)),
    ])

    param_grid = {
    "tfidf__ngram_range": [(1,4),(1, 5)], 
    "tfidf__min_df": [1, 2],                       
    "tfidf__max_df": [0.9, 0.95, 1.0],                     
    "tfidf__sublinear_tf": [False],                                
}
    
    print("Running Grid Search (this will take a while)...")
    grid_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=param_grid,
    cv=3,          
    scoring='f1_macro', 
    n_jobs=2,  
    verbose=3       
)
    grid_search.fit(x_tune, y_tune)
    print("\nBest Parameters found:", grid_search.best_params_)
    
    best_svm = grid_search.best_estimator_
    best_svm.fit(x_train, y_train)          
    results_df["svm_pred"] = best_svm.predict(x_test)

    evaluate_model("VADER", results_df["label"], results_df["vader_pred"])
    evaluate_model("SVM", results_df["label"], results_df["svm_pred"])
    
    print(f"CryptoBert")
    print(f"  Accuracy: 0.5560")
    print(f"  Macro AVG F1 Score: 0.5579")
    print(f"  Macro AVG Precision: 0.4658")
    print(f"  Macro AVG Recall: 0.6044")

    pickle.dump(best_svm, open("../data/models/svm.pkl", "wb"))

Running VADER...
Training & running SVM...
Running Grid Search (this will take a while)...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
[CV 1/3] END tfidf__max_df=0.9, tfidf__min_df=1, tfidf__ngram_range=(1, 4), tfidf__sublinear_tf=False;, score=0.519 total time=  53.7s
[CV 2/3] END tfidf__max_df=0.9, tfidf__min_df=1, tfidf__ngram_range=(1, 4), tfidf__sublinear_tf=False;, score=0.520 total time=  57.0s
[CV 3/3] END tfidf__max_df=0.9, tfidf__min_df=1, tfidf__ngram_range=(1, 4), tfidf__sublinear_tf=False;, score=0.519 total time=  49.7s
[CV 1/3] END tfidf__max_df=0.9, tfidf__min_df=1, tfidf__ngram_range=(1, 5), tfidf__sublinear_tf=False;, score=0.520 total time= 1.1min
[CV 2/3] END tfidf__max_df=0.9, tfidf__min_df=1, tfidf__ngram_range=(1, 5), tfidf__sublinear_tf=False;, score=0.521 total time= 1.1min
[CV 3/3] END tfidf__max_df=0.9, tfidf__min_df=1, tfidf__ngram_range=(1, 5), tfidf__sublinear_tf=False;, score=0.519 total time= 1.2min
[CV 1/3] END tfidf__max_df=0.9, tfidf_

# Model Training and Volatility prediction

The goal of this phase is to build a predictive "Early Warning System" for Bitcoin market volatility. <br>
Rather than attempting to predict the direction of the price we are focused on predicting market explosiveness.

Therefore, i define a "Volatility Spike" as an hour where price fluctuations (High-Low range) fall into the top 33.3% of historical moves. <br>
Successfully identifying these regimes allows for advanced risk management, such as tightening stop-losses or exiting leveraged positions before a potential crash.

Our feature set is built on these pillars:

Market Momentum: 1, 3, 6, and 12-hour lags of volatility and returns. This captures the so called "Volatility Clustering", which is the tendency for big moves to follow big moves.
Social Sentiment: We utilize Sentiment Polarity (The "war" between bulls and bears) and the Post Spike Ratio (sudden surges in Reddit chatter volume).
Temporal Context: By including the hour_of_day, we allow the model to learn that volatility behaves differently during the New York Stock Exchange open than it does during other sessions.

## Initial Findings and Issues

Initially, i attempted a Multiclass (3 classes) approach, categorizing the market into three states: Low (0), Neutral (1), and High (2).<br>
While the models successfully identified periods of dead-quiet volatility (Class 0 F1-score: ~0.88) and violent market spikes (Class 2 F1-score: ~0.62), <br>
they completely failed to predict the "Neutral" market regime (Class 1 F1-score: ~0.30).

## The proposed solution

To maximize the predictive power of our sentiment features and align with our "alarm" objective, we changed the architecture from Multiclass to Binary. <br>
Instead of forcing the model to guess three states, we merged the bottom 66% of market volatility into a single class, isolating the top 33% as our target:

Class 0 ("Normal Market"): The bottom 66.6% of volatility (Low + Neutral). <br>
Class 1 ("Volatility Spike"): The top 33.3% of volatility (High).

By removing the noisy boundary between Low and Neutral, we allow the models to focus strictly on the extreme outliers, the events that actually trigger liquidations.

# MULTICLASS CODE

In [2]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_sample_weight

def train_and_evaluate(model, x, y, tss, model_name):
    """
    Train and evaluate a classifier using time-series cross-validation.
    Feature importances are averaged across all folds.
    """
    fold = 1
    all_importances = []

    for train_index, test_index in tss.split(x):
        print(f"--- Fold {fold} ---")
        x_train, x_test = x.iloc[train_index], x.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        model.fit(x_train, y_train)

        y_pred = model.predict(x_test)
        print(classification_report(y_test, y_pred, zero_division=0))

        all_importances.append(model.feature_importances_)
        fold += 1

    mean_importances = np.mean(all_importances, axis=0)
    feature_importance_df = (
        pd.DataFrame({"Feature": x.columns, "Importance": mean_importances})
        .sort_values(by="Importance", ascending=False)
        .reset_index(drop=True)
    )

    print(f"\nTop 15 Features — {model_name} (avg. across folds)")
    print(feature_importance_df.head(15).to_string(index=False)) 

def main():
    df = pd.read_csv("../data/clean_reddit/final_submission_clean.csv")
    df.set_index("index", inplace=True)
    
    y = df["BTC_vol_target"].astype(int)

    to_drop_cols = ["BTC_vol_target", "ETH_vol_target", "SOL_vol_target"]
    x = df.drop(columns=to_drop_cols)

    tss = TimeSeriesSplit(n_splits=5)

    model_xgb = xgb.XGBClassifier(
        objective="multi:softprob",
        max_depth=5,
        learning_rate=0.05,
        random_state=42,
        eval_metric='logloss',
    )
    model_rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        random_state=42,
        class_weight="balanced",
    )
    model_lgb = lgb.LGBMClassifier(
        objective="multiclass",
        max_depth=3,
        random_state=42,
        class_weight="balanced",
        verbosity=-1,
    )

    print("Training XGBoost Classifier")
    train_and_evaluate(model_xgb, x, y, tss, "XGBoost")

    print("Training Random Forest Classifier")
    train_and_evaluate(model_rf, x, y, tss, "Random Forest")

    print("Training LightGBM Classifier")
    train_and_evaluate(model_lgb, x, y, tss, "LightGBM")

if __name__ == "__main__":
    main()

Training XGBoost Classifier
--- Fold 1 ---
              precision    recall  f1-score   support

           0       0.71      0.57      0.63       650
           1       0.38      0.40      0.39       472
           2       0.43      0.57      0.49       334

    accuracy                           0.52      1456
   macro avg       0.51      0.51      0.51      1456
weighted avg       0.54      0.52      0.52      1456

--- Fold 2 ---
              precision    recall  f1-score   support

           0       0.82      0.23      0.36       371
           1       0.35      0.28      0.31       408
           2       0.58      0.89      0.71       677

    accuracy                           0.55      1456
   macro avg       0.59      0.47      0.46      1456
weighted avg       0.58      0.55      0.51      1456

--- Fold 3 ---
              precision    recall  f1-score   support

           0       0.55      0.55      0.55       475
           1       0.43      0.47      0.45       540
  

# BINARY CODE

In [4]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import classification_report

def train_and_evaluate(model, x, y, tss, model_name):
    """
    Train and evaluate a classifier using time-series cross-validation.
    Feature importances are averaged across all folds.
    """
    fold = 1
    all_importances = []

    for train_index, test_index in tss.split(x):
        print(f"--- Fold {fold} ---")
        x_train, x_test = x.iloc[train_index], x.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        model.fit(x_train, y_train)

        y_pred = model.predict(x_test)
        print(classification_report(y_test, y_pred, zero_division=0))

        all_importances.append(model.feature_importances_)
        fold += 1

    mean_importances = np.mean(all_importances, axis=0)
    feature_importance_df = (
        pd.DataFrame({"Feature": x.columns, "Importance": mean_importances})
        .sort_values(by="Importance", ascending=False)
        .reset_index(drop=True)
    )

    print(f"\nTop 15 Features — {model_name} (avg. across folds)")
    print(feature_importance_df.head(15).to_string(index=False)) 

def main():
    df = pd.read_csv("../data/clean_reddit/final_submission_clean.csv")
    df.set_index("index", inplace=True)
    
    df["BTC_vol_target_binary"] = df["BTC_vol_target"].apply(lambda x: 1 if int(x) == 2 else 0)
    y = df["BTC_vol_target_binary"]

    to_drop_cols = ["BTC_vol_target", "ETH_vol_target", "SOL_vol_target", "BTC_vol_target_binary"]
    x = df.drop(columns=to_drop_cols)

    tss = TimeSeriesSplit(n_splits=5, gap=24)

    model_xgb = xgb.XGBClassifier(
        objective="binary:logistic",
        max_depth=5,
        learning_rate=0.05,
        random_state=42,
        eval_metric='logloss',
        scale_pos_weight=2
    )
    model_rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        random_state=42,
        class_weight="balanced",
    )
    model_lgb = lgb.LGBMClassifier(
        objective="binary",
        max_depth=3,
        learning_rate=0.05,
        random_state=42,
        class_weight="balanced",
        verbosity=-1,
    )

    print("Training XGBoost Classifier")
    train_and_evaluate(model_xgb, x, y, tss, "XGBoost")

    print("Training Random Forest Classifier")
    train_and_evaluate(model_rf, x, y, tss, "Random Forest")

    print("Training LightGBM Classifier")
    train_and_evaluate(model_lgb, x, y, tss, "LightGBM")

if __name__ == "__main__":
    main()

/tmp/ipykernel_6688/2350747130.py:44: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["BTC_vol_target_binary"] = df["BTC_vol_target"].apply(lambda x: 1 if int(x) == 2 else 0)


Training XGBoost Classifier
--- Fold 1 ---
              precision    recall  f1-score   support

           0       0.85      0.80      0.82      1122
           1       0.44      0.51      0.47       334

    accuracy                           0.74      1456
   macro avg       0.64      0.66      0.65      1456
weighted avg       0.75      0.74      0.74      1456

--- Fold 2 ---
              precision    recall  f1-score   support

           0       0.84      0.52      0.64       779
           1       0.62      0.89      0.73       677

    accuracy                           0.69      1456
   macro avg       0.73      0.70      0.69      1456
weighted avg       0.74      0.69      0.68      1456

--- Fold 3 ---
              precision    recall  f1-score   support

           0       0.81      0.76      0.79      1015
           1       0.52      0.58      0.55       441

    accuracy                           0.71      1456
   macro avg       0.66      0.67      0.67      1456
w

# BINARY CODE - HYPERPARAMETERS OPTIMIZATION

We have shown that the binary architecture is superior to the multi-class one on standard classification metrics, indicating better class separability. <br> 
However, we are still evaluating the models using a default 0.50 probability threshold.<br>

A 0.50 threshold assumes that a False Positive and a False Negative cost the exact same amount of money, in trading this is generally false.<br>
The following code modifies the hyperparameter search to prioritize dollars profit over standard metrics via three core mechanics:<br>

### Custom Financial Scoring:<br>
We replace generic evaluation metrics (like F1-Score) with a vectorized financial scorer.<br> 
This translates the confusion matrix directly into simulated dollar returns, factoring in explicit leverage adjusted round-trip fees, slippage and losses from false positives.

### Internal Threshold Search: <br>
Standard grid search penalizes models that perform poorly at a 0.50 threshold.<br>
The scorer evaluates each model under an optimal threshold selection policy within each CV fold.

### Risk Profile Alignment: <br>
Because a "Conservative" portfolio requires a fundamentally different architecture than an "Aggressive" one, the grid search is repeated across distinct financial profiles.

In [5]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import pickle
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import make_scorer

PROFILES = {
    "Conservative": {"gain": 0.030, "cost": 0.010},
    "Standard":     {"gain": 0.060, "cost": 0.020},
    "Aggressive":   {"gain": 0.105, "cost": 0.035}
}

LEVERAGE = 3
FEES = 0.0005
SLIPPAGE = 0.0005
RT_COST = 2 * (FEES + SLIPPAGE) * LEVERAGE

def get_financial_scorer(profile_name):
    def financial_payoff_scorer(y_true, y_probs):
        cfg = PROFILES[profile_name]
        
        y_true_np = np.asarray(y_true)
        y_probs_np = np.asarray(y_probs)
        
        thresholds = np.linspace(0.10, 0.90, 40)
        
        predictions = (y_probs_np[:, None] >= thresholds).astype(int)
        y_true_mat = y_true_np[:, None]

        tp = np.sum((y_true_mat == 1) & (predictions == 1), axis=0)
        fp = np.sum((y_true_mat == 0) & (predictions == 1), axis=0)
        total_returns = (tp * cfg["gain"]) - (fp * cfg["cost"]) - ((tp + fp) * RT_COST)

        return np.max(total_returns)

    return make_scorer(
        financial_payoff_scorer, 
        response_method="predict_proba", 
        greater_is_better=True
    )

def main():
    df = pd.read_csv("../data/clean_reddit/final_submission_clean.csv")
    df.set_index("index", inplace=True)
    
    df["BTC_vol_target_binary"] = df["BTC_vol_target"].apply(lambda x: 1 if int(x) == 2 else 0)
    y = df["BTC_vol_target_binary"]

    to_drop_cols = ["BTC_vol_target", "ETH_vol_target", "SOL_vol_target", "BTC_vol_target_binary"]
    x = df.drop(columns=to_drop_cols)
    
    split_idx = int(len(x) * 0.7)
    x_train_val, _ = x.iloc[:split_idx], x.iloc[split_idx:]
    y_train_val, _ = y.iloc[:split_idx], y.iloc[split_idx:] 
    
    num_neg = (y == 0).sum()
    num_pos = (y == 1).sum()
    estimated_weight = num_neg / num_pos

    tss = TimeSeriesSplit(n_splits=5, gap=24)
    
    for profile_name in PROFILES.keys():
        print(f"OPTIMIZATION FOR: {profile_name.upper()} PORTFOLIO")
        
        portfolio_scorer = get_financial_scorer(profile_name)

        # LIGHTGBM TUNING
        print(f"\nTuning LightGBM Classifier")
        lgb_base = lgb.LGBMClassifier(objective="binary", random_state=42, class_weight="balanced", verbosity=-1, n_jobs=1)
        
        lgb_param_grid = {
            'learning_rate': [0.03],           
            'num_leaves': [31, 45, 63], 
            'reg_alpha': [3.0, 5.0, 10.0],   
            'reg_lambda': [2.0, 5.0],     
            'min_data_in_leaf': [20, 50, 100]
        }
        
        lgb_grid = GridSearchCV(estimator=lgb_base, param_grid=lgb_param_grid, cv=tss, scoring=portfolio_scorer, n_jobs=6, verbose=3)
        lgb_grid.fit(x_train_val, y_train_val)
        
        print(f"Best {profile_name} return (LGB CV): {lgb_grid.best_score_:.4f}x capital")
        print("Best Parameters:", lgb_grid.best_params_)

        # XGBOOST TUNING
        print(f"\nTuning XGBoost Classifier")
        xgb_base = xgb.XGBClassifier(objective="binary:logistic", random_state=42, eval_metric='logloss', scale_pos_weight=estimated_weight, n_jobs=1)

        xgb_param_grid = {
            'learning_rate': [0.03],             
            'max_depth': [4, 5, 6],             
            'gamma': [0, 0.5, 1],               
            'min_child_weight': [1, 5, 10],     
            'reg_lambda': [3.0, 5.0, 7.0]
        }
        
        xgb_grid = GridSearchCV(estimator=xgb_base, param_grid=xgb_param_grid, cv=tss, scoring=portfolio_scorer, n_jobs=6, verbose=3)
        xgb_grid.fit(x_train_val, y_train_val)

        print(f"Best {profile_name} return (XGB CV): {xgb_grid.best_score_:.4f}x capital")
        print("Best Parameters:", xgb_grid.best_params_)
        best_lgb = lgb_grid.best_estimator_
        best_xgb = xgb_grid.best_estimator_
        
        lgb_filename = f"../data/models/lgb_{profile_name.lower()}.pkl"
        xgb_filename = f"../data/models/xgb_{profile_name.lower()}.pkl"
        
        pickle.dump(best_lgb, open(lgb_filename, "wb"))
        pickle.dump(best_xgb, open(xgb_filename, "wb"))
        print(f"Successfully saved: {lgb_filename}")
        print(f"Successfully saved: {xgb_filename}")
        
    print("\nOptimization Complete, all profiles updated.")
    
if __name__ == "__main__":
    main()

OPTIMIZATION FOR: CONSERVATIVE PORTFOLIO

Tuning LightGBM Classifier
Fitting 5 folds for each of 54 candidates, totalling 270 fits


/tmp/ipykernel_6688/1236067497.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["BTC_vol_target_binary"] = df["BTC_vol_target"].apply(lambda x: 1 if int(x) == 2 else 0)


[CV 1/5] END learning_rate=0.03, min_data_in_leaf=20, num_leaves=31, reg_alpha=3.0, reg_lambda=2.0;, score=2.008 total time=   0.4s
[CV 1/5] END learning_rate=0.03, min_data_in_leaf=20, num_leaves=31, reg_alpha=3.0, reg_lambda=5.0;, score=2.056 total time=   0.5s
[CV 4/5] END learning_rate=0.03, min_data_in_leaf=20, num_leaves=31, reg_alpha=3.0, reg_lambda=2.0;, score=3.744 total time=   1.4s
[CV 2/5] END learning_rate=0.03, min_data_in_leaf=20, num_leaves=31, reg_alpha=3.0, reg_lambda=2.0;, score=0.624 total time=   1.4s
[CV 2/5] END learning_rate=0.03, min_data_in_leaf=20, num_leaves=31, reg_alpha=3.0, reg_lambda=5.0;, score=0.608 total time=   1.1s
[CV 3/5] END learning_rate=0.03, min_data_in_leaf=20, num_leaves=31, reg_alpha=3.0, reg_lambda=2.0;, score=7.696 total time=   1.3s
[CV 5/5] END learning_rate=0.03, min_data_in_leaf=20, num_leaves=31, reg_alpha=3.0, reg_lambda=2.0;, score=0.560 total time=   2.0s
[CV 1/5] END learning_rate=0.03, min_data_in_leaf=20, num_leaves=31, reg_alp

# BINARY CODE - THRESHOLD OPTIMIZATION

Starting from the hyperparameter-optimized models obtained in the previous stage, we perform post-training decision optimization by transforming probabilistic model outputs into discrete trading actions through an explicit threshold selection process. <br>
Instead of relying on a fixed probability cutoff (e.g. 0.5), we evaluate a predefined grid of candidate thresholds and select the value that maximizes simulated financial performance on a dedicated calibration set.

## Financial Objective Function

Model evaluation is based on a custom payoff function that directly simulates trading profit and loss: <br>

- True positives contribute a positive return proportional to the selected risk profile’s gain parameter
- False positives incur losses proportional to the cost parameter
- All trades are penalized by explicit round-trip transaction costs (fees and slippage, leverage-adjusted)

## Threshold Selection Procedure

For each trained model and risk profile:

1. Predict class probabilities on the calibration dataset  
2. Evaluate a range of candidate probability thresholds  
3. Convert probabilities into binary trading signals for each threshold  
4. Compute total simulated return for each threshold  
5. Select the threshold that maximizes calibration-period return  

The resulting threshold defines the decision boundary used for out-of-sample evaluation.

## Out-of-Sample Evaluation

The selected threshold is then applied unchanged to a held-out test set. <br>
Performance is assessed using:

- Total simulated return  
- Precision and recall of executed trades  
- Number of trades executed  
- Calibration-to-test return consistency  

This separation ensures that threshold tuning is performed strictly out-of-sample relative to final evaluation.

## Risk Profile Conditioning

The optimization is repeated across multiple predefined risk profiles (Conservative, Standard, Aggressive), each encoding different gain-to-loss structures. <br>
This allows the same predictive model to be adapted into distinct trading strategies with varying risk tolerance.

## Summary

This framework decouples:

- Probabilistic prediction (model output)
- Decision policy (threshold)
- Economic evaluation (PnL)

Ensuring that model performance is measured in terms of actual trading returns rather than classic classification metrics.

In [7]:
import numpy as np
import pandas as pd
import pickle
import json

PROFILES = {
    "Conservative": {"gain": 0.030, "cost": 0.010},
    "Standard":     {"gain": 0.060, "cost": 0.020},
    "Aggressive":   {"gain": 0.105, "cost": 0.035},
}

LEVERAGE = 3
FEES = 0.0005
SLIPPAGE = 0.0005
RT_COST = 2 * (FEES + SLIPPAGE) * LEVERAGE

THRESHOLDS = np.linspace(0.10, 0.90, 80)

def payoff_matrix(y_true: np.ndarray, y_probs: np.ndarray, cfg: dict) -> np.ndarray:
    preds = (y_probs[:, None] >= THRESHOLDS).astype(int)
    y_mat = y_true[:, None]

    tp = np.sum((y_mat == 1) & (preds == 1), axis=0)
    fp = np.sum((y_mat == 0) & (preds == 1), axis=0)
    return tp * cfg["gain"] - fp * cfg["cost"] - (tp + fp) * RT_COST

def find_threshold_on_calibration(y_calib: np.ndarray, probs_calib: np.ndarray, cfg: dict) -> tuple[float, float]:
    returns = payoff_matrix(y_calib, probs_calib, cfg)
    best_idx = int(np.argmax(returns))
    return float(THRESHOLDS[best_idx]), float(returns[best_idx])

def evaluate_fixed_threshold(y_true: np.ndarray, probs: np.ndarray, threshold: float, cfg: dict) -> dict:
    preds = (probs >= threshold).astype(int)

    tp = int(np.sum((y_true == 1) & (preds == 1)))
    fp = int(np.sum((y_true == 0) & (preds == 1)))
    tn = int(np.sum((y_true == 0) & (preds == 0)))
    fn = int(np.sum((y_true == 1) & (preds == 0)))

    total = tp * cfg["gain"] - fp * cfg["cost"] - (tp + fp) * RT_COST

    n_pos = tp + fn
    n_long = tp + fp
    precision = tp / n_long if n_long > 0 else 0.0
    recall = tp / n_pos  if n_pos  > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0)

    all_returns = payoff_matrix(y_true, probs, cfg)

    return {
        "threshold":    threshold,
        "total_return": total,
        "precision":    precision,
        "recall":       recall,
        "f1":           f1,
        "n_trades":     n_long,
        "all_returns":  all_returns,
        "thresholds":   THRESHOLDS,
    }

def print_model_report(label: str, model_type: str, threshold: float, calib_ret: float, metrics: dict) -> None:
    deg = metrics["total_return"] / calib_ret if calib_ret != 0 else 0
    print(f"\n{label} {model_type}")
    print(f"  Threshold : {threshold:.3f}")
    print(f"  Calib     : {calib_ret:+.4f}")
    print(f"  Test      : {metrics['total_return']:+.4f} ({deg:+.1%} of calib)")
    print(f"  Trades    : {metrics['n_trades']}")
    print(f"  Precision : {metrics['precision']:.3f} | Recall: {metrics['recall']:.3f} | F1: {metrics['f1']:.3f}")

def main():
    df = pd.read_csv("../data/clean_reddit/final_submission_clean.csv")
    df.set_index("index", inplace=True)

    df["BTC_vol_target_binary"] = df["BTC_vol_target"].apply(lambda x: 1 if int(x) == 2 else 0)
    y = df["BTC_vol_target_binary"]

    to_drop = ["BTC_vol_target", "ETH_vol_target", "SOL_vol_target", "BTC_vol_target_binary"]
    X = df.drop(columns=to_drop)

    n = len(X)
    train_end = int(n * 0.70)
    calib_end = int(n * 0.85)

    X_calib = X.iloc[train_end:calib_end]
    y_calib = y.iloc[train_end:calib_end].to_numpy()

    X_test  = X.iloc[calib_end:]
    y_test  = y.iloc[calib_end:].to_numpy()

    print(f"Calibration set : {len(X_calib):,} samples "f"({y_calib.mean():.1%} positive)")
    print(f"Test set : {len(X_test):,} samples  "f"({y_test.mean():.1%} positive)")

    all_results = {}

    for profile_name, cfg in PROFILES.items():
        all_results[profile_name] = {}

        for model_type in ("lgb", "xgb"):
            path = f"../data/models/{model_type}_{profile_name.lower()}.pkl"
            model = pickle.load(open(path, "rb"))

            probs_calib = model.predict_proba(X_calib)[:, 1]
            best_threshold, calib_ret = find_threshold_on_calibration(y_calib, probs_calib, cfg)

            probs_test = model.predict_proba(X_test)[:, 1]
            test_metrics = evaluate_fixed_threshold(y_test, probs_test, best_threshold, cfg)

            print_model_report(
                label=model_type.upper(),
                model_type = profile_name,
                threshold=best_threshold,
                calib_ret=calib_ret,
                metrics=test_metrics,
            )

            all_results[profile_name][model_type] = {
                "best_threshold": best_threshold,
                "calib_return":   calib_ret,
                "test_metrics":   test_metrics,
            }

    print("\nSUMMARY — TEST SET PERFORMANCE\n")
    print(f"{'Profile':<14} {'LGB':>12} {'XGB':>12}")

    for profile_name, models in all_results.items():
        lgb_r = models["lgb"]["test_metrics"]["total_return"]
        xgb_r = models["xgb"]["test_metrics"]["total_return"]

        print(f"{profile_name:<14} {lgb_r:+12.4f} {xgb_r:+12.4f}")

    threshold_dict = {
    profile: 
    {
        model_type: model_data["best_threshold"]
        for model_type, model_data in all_results[profile].items()
    }
    for profile in all_results
}
    save_path = "../data/models/optimal_thresholds.json"
    with open(save_path, "w") as f:
        json.dump(threshold_dict, f, indent=4)
        
    print(f"\nSaved optimal thresholds to {save_path}")


if __name__ == "__main__":
    main()

Calibration set : 1,310 samples (16.1% positive)
Test set : 1,311 samples  (14.4% positive)

LGB Conservative
  Threshold : 0.576
  Calib     : +0.8320
  Test      : +1.8960 (+227.9% of calib)
  Trades    : 189
  Precision : 0.651 | Recall: 0.651 | F1: 0.651

XGB Conservative
  Threshold : 0.718
  Calib     : +0.7680
  Test      : +2.0720 (+269.8% of calib)
  Trades    : 143
  Precision : 0.762 | Recall: 0.577 | F1: 0.657

LGB Standard
  Threshold : 0.505
  Calib     : +2.7920
  Test      : +4.5040 (+161.3% of calib)
  Trades    : 236
  Precision : 0.564 | Recall: 0.704 | F1: 0.626

XGB Standard
  Threshold : 0.596
  Calib     : +2.5580
  Test      : +4.9080 (+191.9% of calib)
  Trades    : 202
  Precision : 0.629 | Recall: 0.672 | F1: 0.650

LGB Aggressive
  Threshold : 0.566
  Calib     : +5.6540
  Test      : +9.7680 (+172.8% of calib)
  Trades    : 192
  Precision : 0.656 | Recall: 0.667 | F1: 0.661

XGB Aggressive
  Threshold : 0.546
  Calib     : +5.4850
  Test      : +8.8450 (+1

/tmp/ipykernel_6688/1423073881.py:74: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["BTC_vol_target_binary"] = df["BTC_vol_target"].apply(lambda x: 1 if int(x) == 2 else 0)
